# Research 15 — Paired stratified bootstrap of OOF gains

**Hypothesis:** the newest blend gains may be within row-sampling noise. Estimate paired, class-stratified bootstrap intervals for the leading OOF candidates against the established CatBoost/LightGBM reference. This does not retrain models and does not correct for repeated selection on the same folds; it is a quick uncertainty diagnostic, not independent validation. Test is not loaded.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data" / "train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and np.all(probs >= 0), filename
    sums = probs.sum(axis=1, keepdims=True)
    assert np.all(sums > 0), filename
    return probs / sums


candidates = {
    "Current CatBoost/LightGBM": load_probs("segment-oof.csv"),
    "Four-way loyalty × VIP, seed 42": load_probs("segment-four-way-mixture-oof.csv"),
    "80/20 current + VIP × engagement": load_probs("segment-vip-engagement-blend-oof.csv"),
    "Four-way VIP × engagement": load_probs("segment-vip-engagement-four-way-oof.csv"),
    "Four-way loyalty × VIP, seed 2025": load_probs(
        "segment-lightgbm-loyalty-vip-seed-2025-four-way-oof.csv"
    ),
    "80/20 current + balanced-weight LightGBM": load_probs(
        "segment-lightgbm-balanced-weights-blend-oof.csv"
    ),
    "Four-way with balanced-weight LightGBM": load_probs(
        "segment-lightgbm-balanced-weights-fourway-oof.csv"
    ),
    "80/20 current + VIP × region": load_probs("segment-vip-region-balanced-blend-oof.csv"),
    "80/20 current + VIP × region, seed 2025": load_probs(
        "segment-vip-region-seed-2025-blend-oof.csv"
    ),
    "80/20 VIP × region cross-seed average": load_probs(
        "segment-vip-region-crossseed-blend-oof.csv"
    ),
    "80/20 VIP × region three-seed average": load_probs(
        "segment-vip-region-three-seed-average-oof.csv"
    ),
    "80/20 VIP × region four-seed average": load_probs(
        "segment-vip-region-four-seed-average-oof.csv"
    ),
}
predictions = {name: probs.argmax(axis=1) for name, probs in candidates.items()}
base_name = "Current CatBoost/LightGBM"
base_pred = predictions[base_name]
base_score = f1_score(y, base_pred, average="macro")
print(f"Baseline macro F1={base_score:.6f}; rows={len(y):,}")


def macro_f1_from_counts(sample_y, sample_pred):
    confusion = np.bincount(sample_y * 4 + sample_pred, minlength=16).reshape(4, 4)
    true_positive = np.diag(confusion)
    denominator = confusion.sum(axis=0) + confusion.sum(axis=1)
    per_class = np.divide(2 * true_positive, denominator, out=np.zeros(4), where=denominator > 0)
    return per_class.mean()


classes = np.unique(y)
class_indices = [np.flatnonzero(y == label) for label in classes]
rng = np.random.default_rng(42)
n_bootstrap = 300
deltas = {name: np.empty(n_bootstrap) for name in candidates if name != base_name}
for sample_idx in range(n_bootstrap):
    sample = np.concatenate(
        [rng.choice(indices, size=len(indices), replace=True) for indices in class_indices]
    )
    sample_y = y[sample]
    base_f1 = macro_f1_from_counts(sample_y, base_pred[sample])
    for name in deltas:
        candidate_f1 = macro_f1_from_counts(sample_y, predictions[name][sample])
        deltas[name][sample_idx] = candidate_f1 - base_f1

rows = []
for name, values in deltas.items():
    pred = predictions[name]
    rows.append(
        {
            "candidate": name,
            "macro_f1": f1_score(y, pred, average="macro"),
            "weighted_f1": f1_score(y, pred, average="weighted"),
            "mean_bootstrap_delta": values.mean(),
            "delta_ci_2_5pct": np.quantile(values, 0.025),
            "delta_ci_97_5pct": np.quantile(values, 0.975),
            "bootstrap_fraction_delta_positive": (values > 0).mean(),
        }
    )
results = pd.DataFrame(rows).sort_values("macro_f1", ascending=False)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-oof-bootstrap-screen.csv", index=False)
print(
    f"Completed {n_bootstrap} paired, class-stratified resamples. Conditional on existing OOF predictions; no Test access."
)

Baseline macro F1=0.750895; rows=104,000


,candidate,macro_f1,weighted_f1,mean_bootstrap_delta,delta_ci_2_5pct,delta_ci_97_5pct,bootstrap_fraction_delta_positive
10,80/20 VIP × region four-seed average,0.752172,0.765704,0.001248,0.000467,0.001955,0.996667
9,80/20 VIP × region three-seed average,0.751862,0.765462,0.000956,0.000194,0.001711,0.990000
8,80/20 VIP × region cross-seed average,0.751706,0.765361,0.000820,0.000022,0.001563,0.976667
6,80/20 current + VIP × region,0.751552,0.765087,0.000661,-0.000243,0.001455,0.946667
4,80/20 current + balanced-weight LightGBM,0.751550,0.764986,0.000654,-0.000098,0.001437,0.943333
7,"80/20 current + VIP × region, seed 2025",0.751547,0.765136,0.000674,-0.000162,0.001458,0.956667
5,Four-way with balanced-weight LightGBM,0.751479,0.764955,0.000579,-0.000181,0.001358,0.923333
1,80/20 current + VIP × engagement,0.751447,0.765293,0.000542,-0.000008,0.001150,0.973333
2,Four-way VIP × engagement,0.751386,0.765039,0.000459,-0.000198,0.001031,0.916667
0,"Four-way loyalty × VIP, seed 42",0.751321,0.765050,0.000439,-0.000112,0.001018,0.896667


Completed 300 paired, class-stratified resamples. Conditional on existing OOF predictions; no Test access.


## Interpretation limits

The paired resampling keeps labels and predictions together and preserves each class's sample count. These intervals measure row-level variation conditional on these OOF predictions only: they do not capture training-set, fold, or model-selection uncertainty. Because the same three folds were used repeatedly, even an interval above zero is not a promotion criterion.